# Can we do kingdoms

In [ ]:
#| default_exp game/kingdom

## The politcs of Geography
This is part of a larger game design project where I create a board game generator for my son. I created code that can load or maps with different elevations. from there we can run weather and climate simulators (as well as some terraforming to make sure that rivers wind up going somewhere). The next step is to try to create countries.

At some level, the mistakes are more interesting than the results. Because I use water as what matters, you can see why Sacramento became the capital of California. By no means does this have any historical or real accuracy. but I do think life is better with cool pictures

In [ ]:
#| export
import sys
import math
import numpy as np
import pandas as pd
import random
from fastcore.basics import patch
import heapq # for shortest path
from importlib import resources

In [ ]:
#| export
#This is kitchen sink approach to the library. I just didn't know what I needed

from HexMagic.plot.primitives import  MapCord , PrimitiveDemo
from HexMagic.plot.hex import Hex
from HexMagic.styles import StyleCSS,  SVGBuilder

from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord 
from HexMagic.primitives import HexGrid, HexPosition ,  HexRegion , windy_edge , unique_windy_edge
from HexMagic.terrain import Terrain
from HexMagic.voronoi import generate_plate_terrain
Terrain.fromSeeds = generate_plate_terrain
from HexMagic.climate import ClimatePreset, Climate, TerraDemo
from HexMagic.geology import Geology, DrainageBasins, Watershed


In [ ]:
#| export
from HexMagic.game.settlement import CountryFlag

In [ ]:
#| export
from HexMagic.weather import TerraDemo
from HexMagic.geology import  SoilSystem, DrainageBasins, Geology

Kinddom design

`
def kingdom:
    settlements:[int] # list of cities with [0] being the capital
    hexes:HexRegion
    

    @classmethod
    def start() ->[cls]:
    """
        This returns a list of kingdoms
        we find watersheds and climates that would work and the best hex which isn't necessarily on the coast but probably close
        we then add that hex as our city and the entire watershed as a region
    """

    def expand(is_peaceful=True):
        """
        on the is peaceful setting we check adjacent watersheds that
        1 aren't occupied
        2 are reasonably approachable that is to say we don't have to cross things like rough dessert or high mountains to get to them
        we add those
        """

    def determine_yield()
        """
        we need some way given the hexes to figure out the food
        """
`

In [ ]:
#| export
from typing import NamedTuple

class WatershedFlow(NamedTuple):
    """ a place holder so we don't recompute flows which can be expensive """
    watershed: Watershed
    flow: float


   

In [ ]:
??CountryFlag.encode

In [ ]:
#| export
class TradeRoute:
    """ going to figure how to link up what we build"""

    def __init__(self, path: [HexPosition],  origin = 0, cost: float=0,  name: str=""):
        self.path = path
        self.cost = cost
        self.name = name
        self.origin = origin

    def destination(self)->HexPosition:
        return self.path[-1]

    @staticmethod
    def decode(s: str) -> 'TradeRoute':
        """Decode TradeRoute from string format."""
        name = ""
        origin = 0
        cost = 0.0
        path = []
        
        for line in s.strip().split('\n'):
            if ':' not in line:
                continue
            key, val = line.split(':', 1)
            if key == 'name':
                name = val
            elif key == 'origin':
                origin = int(val)
            elif key == 'cost':
                cost = float(val)
            elif key == 'path':
                path = [int(h) for h in val.split(',') if h]
        
        return TradeRoute(path=path, origin=origin, cost=cost, name=name)


In [ ]:
#| export
@patch
def encode(self: TradeRoute) -> str:
    """Encode TradeRoute to string format."""
    path_str = ','.join(str(h) for h in self.path)
    lines = [
        f"name:{self.name}",
        f"origin:{self.origin}",
        f"cost:{self.cost}",
        f"path:{path_str}"
    ]
    return '\n'.join(lines)

In [ ]:
#| export
@patch
def encode(self: HexRegion) -> str:
    """Encode HexRegion as comma-separated hex indices."""
    return ','.join(str(h) for h in sorted(self.hexes))

@staticmethod
def _decode_hexregion(s: str, hexGrid: HexGrid) -> HexRegion:
    """Decode HexRegion from comma-separated hex indices."""
    hexes = set(int(h) for h in s.split(',') if h)
    return HexRegion(hexes=hexes, hexGrid=hexGrid)

HexRegion.decode = _decode_hexregion


In [ ]:
#| export
class Kingdom:
    """ This is a terrority run by a single government"""
    def __init__(self, capital_hex: int, region: HexRegion, world: Geology,countryId:int = 0, flag = None):
        self.settlements = [capital_hex]  # capital is first
        self.region = region
        self.world = world
        self.countryId = countryId
        self.captured:[WatershedFlow] = []
        self.routes:[TradeRoute] = []
        self.flag = flag
        self.countryName = ""

        
    
    @classmethod
    def start(cls, world: Geology, scored:[WatershedFlow],palette_name: str = "husl"):
        terrain = world.terrain
        """Create initial kingdoms from best watersheds."""
        kingdoms = []
        i = 1 # lets reserve 0 for unclaimed and -1 for unavailable

    
        # Generate styles using seaborn palette
        flags = CountryFlag.seaborn(palette_name, levels=len(scored)+2) # just so we can index by 1
        saturation = 0.7

        for score in scored:
            ws = score.watershed
            capital = ws.max_flow_hex()[0]
          
            kingdom = cls(capital, ws.region, world, countryId=i,flag=flags[i])
            i += 1
            kingdom.captured.append(score)
            kingdoms.append(kingdom)
        
        return kingdoms

    @staticmethod
    def decode(s: str, world: Geology) -> 'Kingdom':
        """Decode Kingdom from string format."""
        lines = s.strip().split('\n')
        
        countryId = 0
        countryName = ""
        settlements = []
        flag = None
        region = None
        captured = []
        routes = []
        
        i = 0
        while i < len(lines):
            line = lines[i]
            
            if line.startswith('countryId:'):
                countryId = int(line.split(':', 1)[1])
            elif line.startswith('countryName:'):
                countryName = line.split(':', 1)[1]
            elif line.startswith('settlements:'):
                settlements = [int(x) for x in line.split(':', 1)[1].split(',') if x]
            elif line.startswith('flag:'):
                flag = CountryFlag.decode(line.split(':', 1)[1])
            elif line.startswith('+region'):
                region_lines = []
                i += 1
                while i < len(lines) and not lines[i].startswith('-region'):
                    region_lines.append(lines[i])
                    i += 1
                region = HexRegion.decode('\n'.join(region_lines), world.terrain.hexGrid)
            elif line.startswith('+captured:'):
                i += 1
                while i < len(lines) and not lines[i].startswith('-captured'):
                    if lines[i].startswith('+watershed:'):
                        flow = float(lines[i].split(':', 1)[1])
                        ws_lines = []
                        i += 1
                        while i < len(lines) and not lines[i].startswith('-watershed'):
                            ws_lines.append(lines[i])
                            i += 1
                        ws = Watershed.decode('\n'.join(ws_lines), world.terrain)
                        captured.append(WatershedFlow(ws, flow))
                    i += 1
            elif line.startswith('+routes:'):
                i += 1
                route_lines = []
                while i < len(lines) and not lines[i].startswith('-routes'):
                    if lines[i] == '---':
                        if route_lines:
                            routes.append(TradeRoute.decode('\n'.join(route_lines)))
                        route_lines = []
                    else:
                        route_lines.append(lines[i])
                    i += 1
            i += 1
        
        kingdom = Kingdom(settlements[0] if settlements else 0, region, world, countryId, flag)
        kingdom.settlements = settlements
        kingdom.countryName = countryName
        kingdom.captured = captured
        kingdom.routes = routes
        
        return kingdom




In [ ]:
#| export
@patch
def encode(self: Kingdom) -> str:
    """Encode Kingdom to string format."""
    lines = [
        f"countryId:{self.countryId}",
        f"countryName:{self.countryName}",
        f"settlements:{','.join(str(s) for s in self.settlements)}",
    ]
    
    # Flag
    if self.flag:
        lines.append(f"flag:{self.flag.encode()}")
    
    # Region
    lines.append("+region")
    lines.append(self.region.encode())
    lines.append("-region")
    
    # Captured watersheds - use Watershed.encode()
    if self.captured:
        lines.append(f"+captured:{len(self.captured)}")
        for wf in self.captured:
            lines.append(f"+watershed:{wf.flow}")
            lines.append(wf.watershed.encode())
            lines.append("-watershed")
        lines.append("-captured")
    
    # Routes
    if self.routes:
        lines.append(f"+routes:{len(self.routes)}")
        for route in self.routes:
            lines.append(route.encode())
            lines.append("---")
        lines.append("-routes")
    
    return '\n'.join(lines)


In [ ]:
#| export
class GameBoard:
    """ This lets us keep track of many of the games global properties"""

    def __init__(self,terrain,top_n=3,year=1900,gender=None):
        self.terrain = terrain
        self.world = Geology(terrain,plates=[])
        self.shedScores = [WatershedFlow(ws, ws.max_flow_hex()[1]) for ws in self.world.basins.sheds]
        self.shedScores.sort(key=lambda x: x[1], reverse=True)

        watershed_map = np.full(len(terrain.elevations), -1)  # -1 = no watershed
        for score in enumerate(self.shedScores):
            ws_id, watFlow = score
            watershed = watFlow[0]
            for hex_idx in watershed.region.hexes:
                watershed_map[hex_idx] = ws_id

        self.watershed_map = watershed_map

        # mark the map where we can have things
        terrain = self.terrain
        countries = np.zeros(len(terrain.elevations))
        for i in range(len(terrain.elevations)):
            if terrain.elevations[i] < 1:
                countries[i] = -1
            if terrain.elevations[i] > terrain.elevationDelta * (len(terrain.colorLevels)-2):
                countries[i] = -1
        
        if len(self.world.basins.sheds) < 0:
            if debug:
                print("no water")
            self.kingdoms = []
            return

        kingdoms = Kingdom.start(self,self.shedScores[:top_n])
        for i, country in enumerate(kingdoms):

            country.countryName =  GameBoard.completeCountry(country.flag.name,country.flag.countryPrefix)
            for h in country.region:
                countries[h] = country.countryId

        terrain.fields["country"] = countries

        self.kingdoms = kingdoms

    @classmethod
    def completeCountry(cls, name ,place, pattern=None, descriptor=None):
        

        # Possessive patterns
        PATTERNS = [
            "{name}'s {place}",      # Karl's Kingdom
            "{place} of {name}",     # Kingdom of Karl
            "{name} {place}",        # Karl Kingdom
        ]

        if not name:
            return ""
        
        # Choose pattern
        if pattern is not None and 0 <= pattern < len(PATTERNS):
            template = PATTERNS[pattern]
        else:
            template = random.choice(PATTERNS)
        
        return template.format(name=name, place=place)
    
    @staticmethod
    def decode(s: str) -> 'GameBoard':
        """Decode GameBoard from string format."""
        lines = s.strip().split('\n')
        
        world = None
        kingdoms = []
        
        i = 0
        while i < len(lines):
            line = lines[i]
            
            if line.startswith('+world'):
                world_lines = []
                i += 1
                while not lines[i].startswith('-world'):
                    world_lines.append(lines[i])
                    i += 1
                world = Geology.decode('\n'.join(world_lines))
            
            elif line.startswith('+kingdoms:'):
                i += 1
                while not lines[i].startswith('-kingdoms'):
                    if lines[i].startswith('+kingdom'):
                        kingdom_lines = []
                        i += 1
                        while not lines[i].startswith('-kingdom'):
                            kingdom_lines.append(lines[i])
                            i += 1
                        kingdoms.append(Kingdom.decode('\n'.join(kingdom_lines), world))
                    i += 1
            
            i += 1
        
        # Create GameBoard without triggering __init__
        board = object.__new__(GameBoard)
        board.terrain = world.terrain
        board.world = world
        board.kingdoms = kingdoms
        
        # Reconstruct derived fields
        board.shedScores = [WatershedFlow(ws, ws.max_flow_hex()[1]) for ws in world.basins.sheds]
        board.shedScores.sort(key=lambda x: x[1], reverse=True)
        
        # Rebuild watershed_map
        watershed_map = np.full(len(board.terrain.elevations), -1)
        for ws_id, watFlow in enumerate(board.shedScores):
            watershed = watFlow.watershed
            for hex_idx in watershed.region.hexes:
                watershed_map[hex_idx] = ws_id
        board.watershed_map = watershed_map
        
        return board

In [ ]:
@patch
def encode(self: GameBoard) -> str:
    """Encode GameBoard to string format."""
    lines = []
    
    # We encode world (which contains terrain and basins)
    lines.append("+world")
    lines.append(self.world.encode())
    lines.append("-world")
    
    # Kingdoms
    lines.append(f"+kingdoms:{len(self.kingdoms)}")
    for kingdom in self.kingdoms:
        lines.append("+kingdom")
        lines.append(kingdom.encode())
        lines.append("-kingdom")
    lines.append("-kingdoms")
    
    return '\n'.join(lines)

#| export
I would like to encode and decode GameBoard

#| export
@patch
def cityName(self:GameBoard, name , pattern=None, descriptor=None, use_suffix=None):
    # Dictionary of place descriptors organized by first letter
    SETTLEMENT_DESCRIPTORS = {
        'A': ['Acres', 'Arbor', 'Ashton', 'Auburn', 'Avon', 'Aldridge', 'Ashford', 'Aston'],
        'B': ['Bay', 'Beach', 'Bridge', 'Brook', 'Burg', 'Borough', 'Bluff', 'Bend'],
        'C': ['City', 'Cove', 'Creek', 'Crest', 'Crossing', 'Center', 'Cape', 'Corners'],
        'D': ['Dale', 'Dell', 'Dunes', 'Down', 'Dock', 'Delta', 'Downs', 'Den'],
        'E': ['End', 'Edge', 'Estates', 'Elms', 'Enclave', 'Evergreen', 'East', 'Elm'],
        'F': ['Falls', 'Field', 'Fields', 'Ford', 'Forest', 'Fort', 'Forks', 'Ferry'],
        'G': ['Glen', 'Glade', 'Green', 'Grove', 'Gate', 'Gardens', 'Groves', 'Gap'],
        'H': ['Harbor', 'Haven', 'Heights', 'Hill', 'Hills', 'Hollow', 'Heath', 'Hurst'],
        'I': ['Isle', 'Island', 'Inlet', 'Inn', 'Ironworks', 'Ivy', 'Isles', 'Inches'],
        'J': ['Junction', 'Jetty', 'Juncture', 'Junction', 'Jamestown', 'Jardin', 'Jct', 'Joya'],
        'K': ['Key', 'Knoll', 'Knolls', 'Keep', 'Keystone', 'Kingswood', 'Kirk', 'Knolle'],
        'L': ['Lake', 'Landing', 'Lawn', 'Ledge', 'Lock', 'Lodge', 'Lagoon', 'Lynn'],
        'M': ['Manor', 'Meadow', 'Meadows', 'Mill', 'Mills', 'Mount', 'Moor', 'Mountain'],
        'N': ['North', 'Nook', 'Narrows', 'Neck', 'Nest', 'Newton', 'New', 'Notch'],
        'O': ['Oaks', 'Orchard', 'Overlook', 'Outpost', 'Outlet', 'Oak', 'Oasis', 'Old'],
        'P': ['Park', 'Pines', 'Plains', 'Point', 'Pond', 'Port', 'Plaza', 'Pass'],
        'Q': ['Quarry', 'Quarter', 'Quay', 'Queen', 'Quarters', 'Quayside', 'Quest', 'Quince'],
        'R': ['Ridge', 'River', 'Rock', 'Run', 'Ranch', 'Rapids', 'Reach', 'Rest'],
        'S': ['Springs', 'Shore', 'Shores', 'South', 'Station', 'Summit', 'Shire', 'Side'],
        'T': ['Town', 'Terrace', 'Trace', 'Trail', 'Township', 'Tower', 'Thicket', 'Timber'],
        'U': ['Union', 'Uplands', 'Upper', 'Underwood', 'Unity', 'University', 'Upton', 'Utopia'],
        'V': ['Vale', 'Valley', 'View', 'Villa', 'Village', 'Vista', 'Ville', 'Vineyards'],
        'W': ['West', 'Water', 'Waters', 'Way', 'Wells', 'Wood', 'Woods', 'Wick'],
        'X': ['Xanadu', 'X-Roads', 'Xing', 'Xavier', 'Xeric', 'Xenia', 'Xenophon', 'Xyst'],
        'Y': ['Yard', 'Yonder', 'York', 'Yards', 'Yew', 'Yews', 'Yale', 'Yarmouth'],
        'Z': ['Zone', 'Zenith', 'Zephyr', 'Zion', 'Zinc', 'Zodiac', 'Zona', 'Zuni'],
    }

    # Naming patterns for settlements
    PATTERNS = [
        "{name}ville",           # Karlville
        "{name}ton",             # Karlton
        "{name}burg",            # Karlburg
        "{name}wood",            # Karlwood
        "{name} {place}",        # Karl Creek
        "{name}'s {place}",      # Karl's Crossing
        "{place} of {name}",     # City of Karl
        "New {name}",            # New Karl
        "Old {name}",            # Old Karl
        "Little {name}",         # Little Karl
        "Upper {name}",          # Upper Karl
        "Lower {name}",          # Lower Karl
        "East {name}",           # East Karl
        "West {name}",           # West Karl
        "North {name}",          # North Karl
        "South {name}",          # South Karl
    ]

    # Shorter patterns for descriptors (avoid double suffixes)
    DESCRIPTOR_PATTERNS = [
        "{name} {place}",        # Karl Creek
        "{name}'s {place}",      # Karl's Crossing
        "{place} of {name}",     # City of Karl
    ]

    if not name:
        return ""
    
    first_letter = name[0].upper()
    
    # Get possible descriptors for this letter
    descriptors = SETTLEMENT_DESCRIPTORS.get(first_letter, ['Place', 'Point', 'Plaza'])
    
    # Decide whether to use suffix or descriptor pattern
    if use_suffix is None:
        use_suffix = random.choice([True, False])
    
    if use_suffix:
        # Use simple suffix patterns (first 7 patterns)
        patterns = PATTERNS[:7]
        if pattern is not None and 0 <= pattern < len(patterns):
            template = patterns[pattern]
        else:
            template = random.choice(patterns)
        
        if "{place}" in template:
            place = descriptor if descriptor and descriptor in descriptors else random.choice(descriptors)
            return template.format(name=name, place=place)
        else:
            return template.format(name=name)
    else:
        # Use directional/size prefix patterns (last 9 patterns)
        patterns = PATTERNS[7:]
        if pattern is not None and 0 <= pattern < len(patterns):
            template = patterns[pattern]
        else:
            template = random.choice(patterns)
        return template.format(name=name)


In [ ]:
def demoKingdoms():

    sampleMap = TerraDemo().aussie_map()
    sampleMap.carve_to_ocean(num_lakes=1)
    sampleWorld = GameBoard(sampleMap,top_n=7)
    
    return sampleWorld

ourBoard = demoKingdoms()
#print(countries)
for country in ourBoard.kingdoms:
    
    print(f"{country.countryName} ruled by {country.flag.name} has {len(country.settlements)}")

aussieText = ourBoard.encode()

In [ ]:
Watershed.decode??

In [ ]:
demoBoard = GameBoard.decode(aussieText)
for country in demoBoard.kingdoms:
    
    print(f"{country.countryName} ruled by {country.flag.name} has {len(country.settlements)}")

Lets think about this some more. for right now a kingdom can only be on one island since we have no way of expanding across ocean. there are two cases
1. the ratio of taken hexes to island is low in which case we do the the wander algorithm.
2. the ratio of taken hexes to island is high in which case we do the grow algorithm

for both algorithm we keep np array which kingdom a hex is in

the wander algorithm
from an aribrary hex of the region(maybe the capital, maybe the centroid we continue out in one of the Hexposition directions (w,e,sw,se,nw,ne) and we stop when we have left the kingdom (there is a version where we store these starting points in the kingdom. If we hit the ocean then we skip this direction. if we hit a claimed hex we skip this direction. if we have more than one direction that worked and is in a watershed we pick the one with the best flow
if none of the hexes we could continue in the direction for each of the remaining hexes with a penalty for stoppig as if the elevation climb is too high or the climate is desert or tundra.
if no hexes are available we stop going for this.



the grow algorithm
this is wander algorithm run in reverse where we start with unclaimed watersheds and see if there are any available kingdoms. for this we just pick the kingdom with the most hexes and add

Since these systems are very bad at working in cube coordinates I have them freshen up on the hex coordinates. 

In [ ]:
!cat ../HexMagic/HexMagic/plot/*.py

In [ ]:
#| export
@patch
def explore(self: GameBoard, countryId) -> Watershed | None:
    country = [x for x in self.kingdoms if x.countryId == countryId][0]
    grid = self.world.terrain.hexGrid
    countries = self.terrain.fields["country"]
    
    candidate_watersheds = []
    
    for direction in HexPosition.directions():
        spot = grid.index_to_hexposition(country.settlements[0])
        place = country.settlements[0]
        
        # Walk in this direction until we leave the kingdom
        while 0 <= place < len(countries) and countries[place] == countryId:
            spot = spot + direction
            place = grid.hexposition_to_index(spot)
        
        # If we hit unclaimed land, check its watershed
        if 0 <= place < len(countries) and countries[place] == 0:
            ws_id = self.watershed_map[place]
            if ws_id >= 0:
                score = self.shedScores[ws_id]
                candidate_watersheds.append(score)
    
    # Return best watershed by flow, or None
    if candidate_watersheds:
        return max(candidate_watersheds, key=lambda x: x[1])
    return None


In [ ]:
#| export
@patch
def expand_kingdoms(self: GameBoard, max_rounds: int = 100):
    """Expand all kingdoms round-robin style until no more growth possible."""
    countries = self.terrain.fields["country"]
    
    for round_num in range(max_rounds):
        any_growth = False
        
        for kingdom in self.kingdoms:
            score = self.explore(kingdom.countryId)
            
            if score is not None:
                any_growth = True
                # Claim all hexes in this watershed
                for hex_idx in score.watershed.region.hexes:
                    countries[hex_idx] = kingdom.countryId
                    kingdom.region.hexes.add(hex_idx)
                kingdom.captured.append(score)
        
        # Stop if no kingdom could grow
        if not any_growth:
            print(f"Expansion complete after {round_num + 1} rounds")
            break
    
    return self.kingdoms


In [ ]:
#| export

@patch
def countries_overlay(self: GameBoard) -> str:
    """Create overlay showing kingdom territories with windy borders.
    """
   
    overlay = ""
    borders = {}  # Shared border cache
    terrain = self.terrain
    grid = terrain.hexGrid

    for  country in self.kingdoms:
        
        
        style = country.flag.kingStyle(f"{country.flag.name}_{country.countryId}")
        #print(StyleCSS.generate([style]))
        grid.builder.add_style(style)

        for path in country.region.trace_perimeter_cached(
            borders,
            style=style,
            f=unique_windy_edge(iterations=2, offset_min=0.05, offset_max=0.15)
        ):
            overlay += path.svg()

    return overlay


### Drawing

In [ ]:
#| export
@patch
def settlements_overlay(self: GameBoard, 
                        capital_size: float = 15,
                        city_size: float = 8) -> str:
    """Create overlay showing settlements with stars for capitals and circles for cities.
    
    Uses kingdom style colors with higher saturation for capitals.
    """
    grid = self.terrain.hexGrid
    overlay = ""
    
    for kingdom in self.kingdoms:
        # Create capital style (more saturated version of kingdom color)
        capital_style = kingdom.flag.contrastStyle(f"capital_{kingdom.countryId}")
        # Increase saturation
        capital_style.properties["fill"] = capital_style.saturate(1.5).properties["fill"]
        grid.builder.add_style(capital_style)
        
        # City style (kingdom color)
        city_style = kingdom.flag.contrastStyle(f"city_{kingdom.countryId}")
        
        grid.builder.add_style(city_style)
        
        for i, settlement_idx in enumerate(kingdom.settlements):
            hex_obj = grid.hexes[settlement_idx]
            cx, cy = hex_obj.center.x, hex_obj.center.y
            
            if i == 0:  # Capital - draw star
                points = []
                for j in range(5):
                    # Outer point
                    angle = (j * 144 - 90) * math.pi / 180
                    points.append(MapCord(
                        cx + capital_size * math.cos(angle),
                        cy + capital_size * math.sin(angle)
                    ))
                    # Inner point
                    angle = (j * 144 + 72 - 90) * math.pi / 180
                    inner_r = capital_size * 0.4
                    points.append(MapCord(
                        cx + inner_r * math.cos(angle),
                        cy + inner_r * math.sin(angle)
                    ))
                
                star_path = MapPath(points, capital_style).closed()
                overlay += star_path.drawClosed()
            else:  # Regular city - draw circle
                overlay += f'\t<circle cx="{cx}" cy="{cy}" r="{city_size}" class="{city_style.name}"/>\n'
    
    return overlay


### Demo

In [ ]:
from HexMagic.geology  import River

@patch
def carve_to_ocean(self: Terrain, num_lakes: int = 5, max_iters: int = 10) -> list[River]:
    """Carve drainage using lowest-cost paths to ocean (or boundary if no ocean)."""
    paths = []
    
    for iteration in range(max_iters):
        minima = find_local_minima(self)
        if len(minima) <= num_lakes:
            print(f"Done at iter {iteration}: {len(minima)} lakes")
            break
        
        minima.sort(key=lambda i: self.elevations[i], reverse=True)
        drain_these = minima[num_lakes:]
        
        for lake_idx in drain_these:
            path = self.find_drainage_path(lake_idx)
            river = River(terrain=self)
            river.tree.create_node(tag="segment", identifier=0, data=path)
            path.reverse()
            river.hexes.update(path)
            paths.append(river)
           
            if len(path) < 2:
                continue
            
            # Interpolate a gentle slope from source to mouth
            start_elev = self.elevations[path[0]]
            end_elev = max(self.elevations[path[-1]], 0)
            
            for i, hex_idx in enumerate(path):
                t = i / (len(path) - 1)  # 0 at source, 1 at mouth
                target = start_elev + t * (end_elev - start_elev)
                # Only lower, never raise
                self.elevations[hex_idx] = min(self.elevations[hex_idx], target)
    
    return paths


In [ ]:
def califorina_place(top_n=5):

    sampleMap = TerraDemo().california_map()
    sampleMap.carve_to_ocean(num_lakes=1)
    sampleMap.hexGrid.adjustRadius(10)
    sampleWorld = GameBoard(sampleMap,top_n=5)
    sampleWorld.expand_kingdoms(max_rounds=50)
    
    return sampleWorld

cali = califorina_place()
caliText = cali.encode()
cali = GameBoard.decode(caliText)

In [ ]:
# Update show_countries to include settlements
def show_countries(game_board: GameBoard, show_settlements: bool = True):
    """Display countries overlay on the terrain."""
    terrain = game_board.terrain
    grid = terrain.hexGrid
    
    grid.builder.layers = []
    terrain.colorMap()
    grid.update()
    grid.builder.adjust("countries", game_board.countries_overlay())
    grid.builder.adjust("water", game_board.world.basins.draw_watersheds())
    if show_settlements:
        grid.builder.adjust("settlement",game_board.settlements_overlay())
    
    return grid.builder.show()

In [ ]:
show_countries(cali,  show_settlements=True)

In [ ]:
def demoKingdoms():

    board =  GameBoard.decode(aussieText)
    board.terrain.carve_to_ocean(num_lakes=1)
    board.terrain.hexGrid.adjustRadius(10)
    
    return board

islandNation = demoKingdoms()
islandNation.expand_kingdoms(max_rounds=50)
show_countries(islandNation,  show_settlements=True)

## Trade

So the next major way that cities are formed is through trade. I want to skip ocean based trade for now (which I fully understand is a big mistake, but I don't have a quick idea on how to do harbors. for land based trade.
1. We can compute if two countries are adjacent if we look at one's region and use the outside function
2. The naive way to connect adjacent would be using the path_through_waypoints, but I think we want more of https://en.wikipedia.org/wiki/Dijkstra%27s_algorithm the weight for going into a hex is the elevation climb (and a default for when things are flat or downhill) right now we can't cross water so those paths are off limits.

lets create a few things
1. something that would give us a list of routes between our capitals that are adjacent to each other. Because of the elevation gain algorithim the route to a country could be different that the route from a country
2. an overlay where we use the hexgrid.arrow function to draw these routes
3. figure out if there are good places along these routes to create settlements. they must have a decent watershed and be a certain distance so that the travelers can reload.

In [ ]:
#| export
@patch
def find_adjacent_kingdoms(self: Kingdom, all_kingdoms: list['Kingdom']) -> list[int]:
    """Find kingdoms that share a border with this one."""
    adjacent = []
    outside_hexes = self.region.outside(ring=1)
    
    for i, other in enumerate(all_kingdoms):
        if other is self:
            continue
        
        # Check if any of our outside hexes are in their region
        if any(hex_idx in other.region.hexes for hex_idx in outside_hexes.hexes):
            adjacent.append(i)
    
    return adjacent




In [ ]:
#| export
@patch
def movement_cost(self: GameBoard, from_pos: HexPosition, to_pos: HexPosition, origin: int) -> float:
    """Calculate cost to move from one hex to another using HexPositions.
    
    Args:
        from_pos: Starting HexPosition
        to_pos: Destination HexPosition
        origin: Origin index for converting HexPosition to grid index
    
    Returns infinity for invalid moves (water, out of bounds).
    """
    terrain = self.terrain
    grid = terrain.hexGrid
    
    # Convert to indices to check terrain
    to_idx = grid.hexposition_to_index(to_pos, origin)
    from_idx = grid.hexposition_to_index(from_pos, origin)
    
    # Check bounds
    if to_idx < 0 or to_idx >= len(terrain.elevations):
        return float('inf')
    
    # Can't cross water
    if terrain.elevations[to_idx] < 1:
        return float('inf')
    
    from_elev = terrain.elevations[from_idx]
    to_elev = terrain.elevations[to_idx]
    
    # Base cost
    base_cost = 1.0
    
    # Elevation gain penalty (climbing is expensive)
    elev_diff = to_elev - from_elev
    if elev_diff > 0:
        # Exponential penalty for climbing
        base_cost += elev_diff * 2.0
    
    return base_cost




In [ ]:
#| export
@patch
def find_path_dijkstra(self: GameBoard, start_hex: int, end_hex: int) -> list[HexPosition] | None:
    """Find shortest path using Dijkstra's algorithm with elevation costs."""
    #import heapq this is installed in the header

    class BHeapItem:
        def __init__(self, cost, hex_pos):
            self.cost = cost
            self.hex_pos = hex_pos

        def __lt__(self, other):
            return self.cost < other.cost

    grid = self.terrain.hexGrid
    
    # Convert start to HexPosition (using start as origin)
    start_pos = grid.index_to_hexposition(start_hex, start_hex)  # (0,0,0)
    end_pos = grid.index_to_hexposition(end_hex, start_hex)
    
    # Priority queue: (cost, HexPosition)
    queue = [BHeapItem(0, start_pos)]
    costs = {start_pos: 0}
    came_from = {}
    
    while queue:
        item = heapq.heappop(queue)
        current_cost, current = item.cost, item.hex_pos
        
        if current == end_pos:
            # Reconstruct path
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(start_pos)
            return list(reversed(path))
        
        # Skip if we've found a better path already
        if current_cost > costs.get(current, float('inf')):
            continue
        
        # Check all 6 neighbors
        for direction in HexPosition.directions():
            neighbor = current + direction
            
            cost = self.movement_cost(current, neighbor, start_hex)
            if cost == float('inf'):
                continue
            
            new_cost = current_cost + cost
            
            if new_cost < costs.get(neighbor, float('inf')):
                costs[neighbor] = new_cost
                came_from[neighbor] = current
                heapq.heappush(queue, BHeapItem(new_cost, neighbor))
    
    return None  # No path found

### Demo

In [ ]:
def califorina_place(top_n=5):

    sampleMap = TerraDemo().california_map()
    #sampleMap.carve_to_ocean(num_lakes=1)
    sampleMap.hexGrid.adjustRadius(10)
    sampleWorld = GameBoard(sampleMap,top_n=5)
    sampleWorld.expand_kingdoms(max_rounds=50)
    for country in sampleWorld.kingdoms:
        neighbors = country.find_adjacent_kingdoms(sampleWorld.kingdoms)
        for dest in neighbors:
            origin = country.settlements[0]
            path = sampleWorld.find_path_dijkstra(origin, sampleWorld.kingdoms[dest].settlements[0])
            if path is not None:
                country.routes.append(TradeRoute(path,origin=origin))
        print(f"{country.countryId} has {len(country.routes)} routes")
    
    return sampleWorld

cali = califorina_place()

In [ ]:
cali.terrain.colorMap()
cali.terrain.hexGrid.update()
cali.terrain.builder.show()

In [ ]:
Terrain.carve_to_ocean??

In [ ]:
#| export
@patch
def trade_overlay(self: GameBoard, 
                        capital_size: float = 15,
                        city_size: float = 8) -> str:
    """Create overlay showing trade routes between capitals.
    """
    grid = self.terrain.hexGrid
    overlay = ""
    
    for kingdom in self.kingdoms:
        # Create capital style (more saturated version of kingdom color)
        
        arrow_style = StyleCSS(
            f"arrow_{kingdom.countryId}",
            stroke="#5c4033",  # brown/dirt road color
            stroke_width=2,
            fill="#5c4033"
        )

        grid.builder.add_style(arrow_style)

        for route in kingdom.routes:
            origin = grid.index_to_hexposition(route.origin)
            lastPos = route.path[0]
            for pos in route.path:
                if pos != lastPos:
                    start = grid.hexposition_to_index(lastPos, route.origin)
                    end = grid.hexposition_to_index(pos,route.origin)
                    if start >= 0 and end >= 0:
                        overlay += grid.arrow(start, end, arrow_style)
                    else:
                        print("off map")
                lastPos = pos
        
    
    return overlay


In [ ]:
#| export
@patch
def names_overlay(self: GameBoard) -> str:
    """Create overlay showing kingdom territories with windy borders and labels."""
    terrain = self.terrain
    grid = terrain.hexGrid
    
    
    
    overlay = ""
    for country in self.kingdoms:
        # Add label style
        label_style =  country.flag.labelStyle(f"n_{country.countryId}")
        label_style.properties["stroke"] = "#36454F"
        label_style.properties["fill"] = "none"
        #print(StyleCSS.generate([label_style]))
        grid.builder.add_style(label_style)
        if len(country.region.hexes) > 0:
            # Add country name at centroid
            centroid_idx = country.region.centroid_hex()
            if centroid_idx >= 0 and country.countryName:
                hex_obj = grid.hexes[centroid_idx]
                cx, cy = hex_obj.center.x, hex_obj.center.y
                overlay += f'\t<text x="{cx}" y="{cy}" text-anchor="middle" dominant-baseline="middle" class="{label_style.name}">{country.countryName}</text>\n'

    return overlay


In [ ]:
# Update show_countries to include settlements
def show_countries(game_board: GameBoard, show_settlements: bool = True):
    """Display countries overlay on the terrain."""
    terrain = game_board.terrain
    grid = terrain.hexGrid
    
    grid.builder.layers = []
    terrain.colorMap()
    grid.update()
    grid.builder.adjust("countries", game_board.countries_overlay())
    grid.builder.adjust("water", game_board.world.basins.draw_watersheds())
    if show_settlements:
        grid.builder.adjust("settlement",game_board.settlements_overlay())
        grid.builder.adjust("trade",game_board.trade_overlay())
    
    return grid.builder.show()

In [ ]:
def showClimes(terrain:Terrain,num_lakes=None,top_n=6,show_trade=False,year=1900):
    grid = terrain.hexGrid
    builder = grid.builder
    builder.layers = []

    if num_lakes is not None:
        terrain.carve_to_ocean(num_lakes=num_lakes)
    terrain.hexGrid.adjustRadius(10)
    sampleWorld = GameBoard(terrain,top_n=top_n,year=year)
    sampleWorld.expand_kingdoms(max_rounds=50)
    for country in sampleWorld.kingdoms:
        neighbors = country.find_adjacent_kingdoms(sampleWorld.kingdoms)
        for dest in neighbors:
            origin = country.settlements[0]
            path = sampleWorld.find_path_dijkstra(origin, sampleWorld.kingdoms[dest].settlements[0])
            if path is not None:
                country.routes.append(TradeRoute(path,origin=origin))


    terrain.colorMap()
    terrain.compute_climate()
    
    terrain.terrainCream()

    builder.adjust("climates", terrain.dottedClimate())
    builder.adjust("settlement",sampleWorld.settlements_overlay())
    builder.adjust("countries", sampleWorld.countries_overlay())
    builder.adjust("water", sampleWorld.world.basins.draw_watersheds())
    builder.adjust("names",sampleWorld.names_overlay())
    
    if show_trade:
        builder.adjust("trade",sampleWorld.trade_overlay())

    return builder.show()

In [ ]:
show_countries(cali)

In [ ]:
showClimes(TerraDemo().california_map(),num_lakes=0,top_n=14,year=1980)

In [ ]:

def demo_world(sampleMap,top_n=5,num_lakes= None):
    if num_lakes is not None:
        sampleMap.carve_to_ocean(num_lakes=num_lakes)
    sampleMap.hexGrid.adjustRadius(10)
    sampleWorld = GameBoard(sampleMap,top_n=top_n)
    sampleWorld.expand_kingdoms(max_rounds=50)
    for country in sampleWorld.kingdoms:
        neighbors = country.find_adjacent_kingdoms(sampleWorld.kingdoms)
        for dest in neighbors:
            origin = country.settlements[0]
            path = sampleWorld.find_path_dijkstra(origin, sampleWorld.kingdoms[dest].settlements[0])
            if path is not None:
                country.routes.append(TradeRoute(path,origin=origin))
        print(f"{country.countryId} has {len(country.routes)} routes")
    
    return show_countries(sampleWorld,  show_settlements=True)
demo_world(TerraDemo().aussie_map())


In [ ]:
showClimes(TerraDemo().aussie_map(),num_lakes=0,top_n=14)

In [ ]:
showClimes(TerraDemo().hundred_years_map(),num_lakes=0,top_n=24,year=2011)

In [ ]:
syd = TerraDemo().sydney_map()
syd.hexGrid.adjustRadius(8)
syd.elevationDelta = 60
showClimes(syd,top_n=15,num_lakes=0,year=1954)

In [ ]:
syd = TerraDemo().thirteen_colonies_map()
syd.hexGrid.adjustRadius(12)
demo_world(syd,top_n=7)

In [ ]:
syd = TerraDemo().rio_map()
syd.hexGrid.adjustRadius(12)
demo_world(syd,top_n=17)


Any thoughts on where to go next?


In [ ]:
showClimes(TerraDemo().hundred_years_map(),num_lakes=0,top_n=24)

In [ ]:
def califorina_place(top_n=5):

    sampleMap = TerraDemo().california_map().downsample_climate(0.25)
    sampleMap.carve_to_ocean(num_lakes=1)
    sampleMap.hexGrid.adjustRadius(10)
    sampleWorld = GameBoard(sampleMap,top_n=5)
    sampleWorld.expand_kingdoms(max_rounds=50)
    
    return sampleWorld

cali = califorina_place()